LOAD DATA

In [1]:
import os
import pandas as pd
import numpy as np

base_path = r"C:\Behavioral-Based_Authentication\public_dataset"

dataset = []

for user_id in os.listdir(base_path):
    user_path = os.path.join(base_path, user_id)

    if not os.path.isdir(user_path):
        continue

    for session in os.listdir(user_path):
        session_path = os.path.join(user_path, session)

        if not os.path.isdir(session_path):
            continue

        features = {
            "user": user_id,
            "session": session
        }

        for file in os.listdir(session_path):
            if not file.endswith(".csv"):
                continue

            file_path = os.path.join(session_path, file)

            if os.path.getsize(file_path) == 0:
                continue

            try:
                # baca sebagian
                df = pd.read_csv(file_path, header=None, nrows=2000)

                # kalau 1 kolom → split
                if df.shape[1] == 1:
                    df = df[0].astype(str).str.split(",", expand=True)

                df = df.apply(pd.to_numeric, errors='coerce')
                df = df.dropna()

                if df.empty:
                    continue

                # ambil semua kolom numerik
                numeric_df = df.select_dtypes(include=np.number)

                for col in numeric_df.columns:
                    features[f"{file}_{col}_mean"] = numeric_df[col].mean()
                    features[f"{file}_{col}_std"] = numeric_df[col].std()

            except:
                continue

        dataset.append(features)

# dibuat dataframe supaya ga terlalu besar
final_df = pd.DataFrame(dataset).fillna(0)

print("Final shape:", final_df.shape)
print(final_df.head())

Final shape: (216, 260)
     user            session  Accelerometer.csv_0_mean  \
0  100669   100669_session_1              1.396226e+12   
1  100669  100669_session_10              1.396397e+12   
2  100669  100669_session_11              1.396397e+12   
3  100669  100669_session_12              1.396398e+12   
4  100669  100669_session_13              1.396481e+12   

   Accelerometer.csv_0_std  Accelerometer.csv_1_mean  Accelerometer.csv_1_std  \
0              5831.043429              6.795260e+12             5.832630e+09   
1              5780.785240              3.765136e+12             5.780262e+09   
2              5810.615454              4.038736e+12             5.814216e+09   
3              5779.341305              4.712626e+12             5.780022e+09   
4              5779.470932              2.556817e+12             5.780163e+09   

   Accelerometer.csv_2_mean  Accelerometer.csv_2_std  \
0              1.006690e+14                 0.953363   
1              1.006691e+14 

SIAPIN DATA

In [2]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(final_df["user"])

X = final_df.drop(columns=["user", "session"])

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

MODEL 1 — RANDOM FOREST

In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)

rf_train_acc = rf.score(X_train, y_train)
rf_test_acc = rf.score(X_test, y_test)

print("Train:", rf_train_acc)
print("Test :", rf_test_acc)

rf_acc = rf.score(X_test, y_test)
print("Random Forest Accuracy:", rf_acc)

Train: 1.0
Test : 1.0
Random Forest Accuracy: 1.0


MODEL 2 — XGBOOST

In [5]:
from xgboost import XGBClassifier

xgb = XGBClassifier(eval_metric='mlogloss')
xgb.fit(X_train, y_train)

xgb_acc = xgb.score(X_test, y_test)
print("Accuracy:", xgb_acc)

Accuracy: 0.9545454545454546


MODEL 3 — SVM

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC())
])

svm_model.fit(X_train, y_train)
svm_acc = svm_model.score(X_test, y_test)

print("SVM (fixed):", svm_acc)

SVM (fixed): 0.7272727272727273


Although Random Forest achieved perfect accuracy, this may indicate overfitting due to the high dimensionality of the dataset. XGBoost demonstrated strong and more reliable performance, suggesting better generalization capability. In contrast, SVM struggled due to the complexity and scale of the feature space.

PERBANDINGAN MODEL

In [7]:
print("\n=== MODEL COMPARISON ===")
print("Random Forest :", rf_acc)
print("XGBoost       :", xgb_acc)
print("SVM           :", svm_acc)


=== MODEL COMPARISON ===
Random Forest : 1.0
XGBoost       : 0.9545454545454546
SVM           : 0.7272727272727273


CONFUSION MATRIX (BIAR KELIHATAN ILMIAH)

In [8]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = xgb.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[4 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0]
 [0 0 3 0 0 0 0 0 0]
 [0 0 0 4 0 0 0 0 0]
 [0 0 0 1 7 0 0 0 0]
 [0 0 0 0 0 6 0 0 0]
 [0 0 0 0 0 0 4 0 0]
 [0 0 0 0 0 0 0 7 0]
 [0 0 0 0 0 0 0 1 4]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00         3
           2       1.00      1.00      1.00         3
           3       0.80      1.00      0.89         4
           4       1.00      0.88      0.93         8
           5       1.00      1.00      1.00         6
           6       1.00      1.00      1.00         4
           7       0.88      1.00      0.93         7
           8       1.00      0.80      0.89         5

    accuracy                           0.95        44
   macro avg       0.96      0.96      0.96        44
weighted avg       0.96      0.95      0.95        44



FEATURE IMPORTANCE 

In [9]:
import matplotlib.pyplot as plt
import pandas as pd

importance = xgb.feature_importances_
feat_names = X.columns

feat_imp = pd.DataFrame({
    "feature": feat_names,
    "importance": importance
}).sort_values(by="importance", ascending=False)

print(feat_imp.head(10))

                            feature  importance
152         StrokeEvent.csv_16_mean    0.195963
172           TouchEvent.csv_9_mean    0.134371
6          Accelerometer.csv_3_mean    0.105730
109          ScrollEvent.csv_12_std    0.100544
54          Magnetometer.csv_3_mean    0.065290
232       tempTouchEvent.csv_9_mean    0.061420
191            PinchEvent.csv_7_std    0.057144
56          Magnetometer.csv_4_mean    0.055964
80   OneFingerTouchEvent.csv_9_mean    0.045668
0          Accelerometer.csv_0_mean    0.040456
